In [ ]:
import numpy as np
import numexpr as ne
import matplotlib.pyplot as plt
from numba import njit, prange


# **Let the Compiler Help Us Compute: Practical Compiled Code for Neuroscience**

Vectorized NumPy code is often fast. The more work we can hand over to compiled functions, the less often they have to interact with Python in order to do their work.

In this notebook, we'll look at two powerful tools for doing more computational work outside of Python's runtime:

* `numexpr` — to evaluate array expressions more efficiently
* `numba` — to compile Python functions into machine code

By the end of this notebook, we should be able to:

* Recognize when Python overhead limits performance
* Use `numexpr` to reduce interpreter overhead in array expressions
* Use Numba's `@njit` decorator to compile loop-based code
* Understand compilation overhead and when it matters
* Avoid common pitfalls that prevent compilation



---


# Section 1 — Using `numexpr` to Reduce Python Overhead

NumPy is fast because its core operations run in compiled C code. But every time we write an expression like:

```python
np.arange(10_000_000) + 1 + 2 + 3 + 4 + 5 + 6
```

Python still:

* Parses the expression
* Allocates temporary arrays
* Manages intermediate results


`numexpr` lets you create a micro-function that:

    * Minimizes temporary arrays
    * Uses multiple CPU cores automatically


### Reference

| Code                    | Description                           |
| ----------------------- | ------------------------------------- |
| `import numexpr as ne`  | Import numexpr.                       |
| `ne.evaluate(expr)`     | Evaluate an expression using numexpr. |
| `ne.NumExpr(expr)`      | Pre-compile an expression.            |
| `f(x=x)`                | Execute a compiled expression.        |
| `ne.set_num_threads(n)` | Control number of threads.            |


### Exercises

**Example**: Speed up the calculation below with `ne.evaluate`:

In [ ]:
# Before (keep this cell the same, for later comparison)

x = np.random.random(10_000_000)

y = %time x * 2 + 3
y.sum()

In [ ]:
# After (modify this code cell)

x = np.random.random(10_000_000)

y = %time ne.evaluate('x * 2 + 3')
y.sum()

**Exercise**: Speed up the calculation below with `ne.evaluate`:

In [ ]:
# Before (keep this cell the same, for later comparison)

x = np.random.random(10_000_000)

y = %time 3 * x ** 2 + 2 * x + 1
y.sum()

In [ ]:
# After (modify this code cell)

x = np.random.random(10_000_000)

y = %time 3 * x ** 2 + 2 * x + 1
y.sum()

**Exercise**: Speed up the timed calculation below using `ne.evaluate()`:

In [ ]:
# Before (keep this the same, for comparison)


x = np.random.random(20_000_000)
y = np.random.random(20_000_000)
z = np.random.random(20_000_000)

total = %time x + y + z

plt.figure(figsize=(10, 1.5))
plt.subplot(1, 4, 1); plt.hist(x, bins=201);
plt.subplot(1, 4, 2); plt.hist(y, bins=201);
plt.subplot(1, 4, 3); plt.hist(z, bins=201);
plt.subplot(1, 4, 4); plt.hist(total, bins=201);
plt.tight_layout();

In [ ]:
# After (modify this cell)

x = np.random.random(20_000_000)
y = np.random.random(20_000_000)
z = np.random.random(20_000_000)

total = %time x + y + z

plt.figure(figsize=(10, 1.5))
plt.subplot(1, 4, 1); plt.hist(x, bins=201);
plt.subplot(1, 4, 2); plt.hist(y, bins=201);
plt.subplot(1, 4, 3); plt.hist(z, bins=201);
plt.subplot(1, 4, 4); plt.hist(total, bins=201);
plt.tight_layout();

**Exercise**: Speed up the following calculation with `np.evaluate():

In [ ]:
x = np.random.random(5_000_000)

In [ ]:
%%time
# Before (keep this the same, for comparison)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

In [ ]:
%%time
# After (modify this cell)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

**Exercise**: Let's do it again, but this time with a smaller array.  What is different about the performance?

In [ ]:
x = np.random.random(50)

In [ ]:
%%time
# Before (keep this the same, for comparison)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

In [ ]:
%%time
# After (modify this cell)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

**Exercise**: Let's try the smaller case again, but this time pre-compiling the function using the pattern `f = ne.NumExpr('x + 1'); f(x)`.  How does this affect the performance?



In [ ]:
%%time
# Before (keep this the same, for comparison)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

In [ ]:
%%time
# After (modify this cell)

for _ in range(50):
    x = x * 2 * 2 * 2 * 2

**Exercise**: `numexpr` is great at helping with broadcasting, but is not a full `numpy` replacement.  Let's try it out here, and see if it helps:

In [ ]:
x = np.random.random(10_000_000)

%time np.sum(np.sqrt(x))

In [ ]:
x = np.random.random(10_000_000)

%time np.sum(np.sqrt(x))

---

## Section 2: Just-in-Time Compilation with `@njit`


In Section 1, we improved performance for vectorized array expressions, but not all problems are easy to express with pure NumPy operations.

In neuroscience, we often need:

  - Custom spike detection rules
  - Trial-by-trial logic
  - Nested loops
  - Conditional branching
  - Simulation steps

Pure Python loops are slow because:

* Every iteration runs through the Python interpreter
* Types are dynamic
* Operations are dispatched at runtime

Numba allows us to compile Python functions into machine code using Just-in-Time (JIT) compilation.

With `@njit`, we:

* Keep normal Python syntax
* Restrict ourselves to supported features
* Gain near-C speed

### Reference

| Code                     | Description                        |
| ------------------------ | ---------------------------------- |
| `from numba import njit` | Import JIT decorator.              |
| `@njit`                  | Compile function in nopython mode. |
| `@jit`                   | Compile but allow Python fallback. |
| `f.inspect_types()`      | Show inferred types.               |



### Exercises

For exach of the exercises below, we'll take an existing Python function and add `@njit`, and compare performance.

**Exercise**:  Basic Spike Counter


In [ ]:
## Before (keep this the sam, for later comparison)

def count_spikes(x, threshold):
    count = 0
    for i in range(len(x)):
        if x[i] > threshold:
            count += 1
    return count


In [ ]:
## Call this cell multiple times and check if the performance changes

x = np.random.random(10_000_000)
%time count_spikes(x, 0.1)

In [ ]:
## After (moidfy this cell)

def count_spikes(x, threshold):
    count = 0
    for i in range(len(x)):
        if x[i] > threshold:
            count += 1
    return count



In [ ]:
## Call this cell multiple times and check if the performance changes

x = np.random.random(10_000_000)
%time count_spikes(x, 0.1)

**Exercise**:  Sum of Squares

In [ ]:
# Before (keep this the same, for later comparison)

def sum_of_squares(x):
    total = 0.0
    for i in range(len(x)):
        total += x[i] * x[i]
    return total


In [ ]:
# Time the function here

In [ ]:
# After (add @njit to this function)

def sum_of_squares(x):
    total = 0.0
    for i in range(len(x)):
        total += x[i] * x[i]
    return total


In [ ]:
# Time the function here

**Exercise**: Conditional Logic

In [ ]:
# Before (keep this the same, for later comparison)

def weighted_sum(x, threshold):
    total = 0.0
    for i in range(len(x)):
        if x[i] > threshold:
            total += x[i]
        else:
            total -= x[i]
    return total

In [ ]:
# time it here

In [ ]:
# After (add @njit to this one)

def weighted_sum(x, threshold):
    total = 0.0
    for i in range(len(x)):
        if x[i] > threshold:
            total += x[i]
        else:
            total -= x[i]
    return total

In [ ]:
# time it here

---



# Section 3 — Requesting Parallel Execution with `parallel=True` and `prange`

Now we go one step further: we ask Numba to run loops in parallel across CPU cores.  This is helpful, for example, if we are computing spike counts independently across 200 recording channels. Each channel can be processed independently — which makes it a good candidate for parallelization.

We do this with:

  -  `@njit(parallel=True)`
  -  `prange()` instead of `range()`

However:

* Not all loops can be parallelized.
* Parallel overhead is real.
* Numba does not always parallelize even when we request it.




### Reference 

| Code                              | Description                    |
| --------------------------------- | ------------------------------ |
| `@njit(parallel=True)`            | Enable parallel compilation.   |
| `from numba import prange`        | Import parallel range.         |
| `prange(n)`                       | Parallel loop iterator.        |
| `f.parallel_diagnostics(level=4)` | Show parallel analysis report. |


### Exercises

**Exercise**: Add `(parallel=True)` and `prange()` to this function.  What performance benefit is there?

In [ ]:
def sum_seq(x):
    total = 0.0
    for i in range(len(x)):
        total += x[i]
    return total


In [ ]:
x = np.random.random(50_000_000)
sum_seq(x)


Run `sum_seq.parallel_diagnostics()` and give the output to an AI.  Was Numba able to paralllize this code?

In [ ]:
sum_seq.parallel_diagnostics(level=4)

**Exercise**: Add `(parallel=True)` and `prange()` to this function.  What performance benefit is there?

In [ ]:

def dependent_loop(x):
    for i in range(1, len(x)):
        x[i] = x[i] + x[i - 1]
    return x

In [ ]:
x = np.random.random(50_000_000)
dependent_loop(x)

Run `sum_seq.parallel_diagnostics()` and give the output to an AI.  Was Numba able to paralllize this code?